# 📊 交互式可视化教程

## 📚 简介

这个教程将教你如何创建**交互式图表**，让数据更生动、更易探索！

**工具**: Plotly Express

**优势**:
- ✅ 可以缩放、平移
- ✅ 悬停显示详细信息
- ✅ 动态筛选
- ✅ 可导出为HTML

---

## 🚀 第一步：安装和导入库

In [ ]:
# 安装plotly（如果还没有安装）
# !pip install plotly

import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 设置pandas显示
pd.set_option('display.max_columns', None)

print("✅ 交互式可视化库已加载！")

## 📍 第二步：创建示例数据

In [ ]:
# 创建示例数据：三个城市的温度和NDVI
np.random.seed(42)

cities = ['北京', '上海', '广州']
months = ['1月', '2月', '3月', '4月', '5月', '6月', 
          '7月', '8月', '9月', '10月', '11月', '12月']

data = []
for city in cities:
    for month in months:
        # 模拟温度（北方城市变化更大）
        if city == '北京':
            temp_base = 5 + (int(month[:-1]) - 1) * 2.5
            temp_noise = np.random.normal(0, 2)
        elif city == '上海':
            temp_base = 10 + (int(month[:-1]) - 1) * 2
            temp_noise = np.random.normal(0, 1.5)
        else:  # 广州
            temp_base = 15 + (int(month[:-1]) - 1) * 1.5
            temp_noise = np.random.normal(0, 1)
        
        temperature = temp_base + temp_noise
        
        # 模拟NDVI（夏季高）
        month_idx = int(month[:-1])
        if 5 <= month_idx <= 9:  # 夏季
            ndvi_base = 0.6 + np.random.normal(0, 0.05)
        else:
            ndvi_base = 0.3 + np.random.normal(0, 0.05)
        
        ndvi = max(0, min(1, ndvi_base))
        
        data.append({
            '城市': city,
            '月份': month,
            '温度(°C)': round(temperature, 1),
            'NDVI': round(ndvi, 3)
        })

df = pd.DataFrame(data)

print("✅ 示例数据已创建！")
print(f"数据量: {len(df)} 行")
print("\n数据预览:")
print(df.head(10))

## 📈 可视化1：交互式折线图

In [ ]:
# 创建交互式折线图：温度变化趋势
fig = px.line(df, 
              x='月份', 
              y='温度(°C)', 
              color='城市',
              title='三城市温度变化趋势 (2023)',
              markers=True,
              line_shape='spline',
              labels={'月份': '月份', '温度(°C)': '温度 (°C)'})

# 更新布局
fig.update_layout(
    hovermode='x unified',
    xaxis_title='月份',
    yaxis_title='温度 (°C)',
    title_font_size=16,
    template='plotly_white'
)

# 显示图表
fig.show()

print("✅ 折线图已生成！")
print("\n交互功能:")
print("  - 鼠标悬停查看详细数据")
print("  - 点击图例隐藏/显示线条")
print("  - 双击重置缩放")
print("  - 拖拽平移，滚轮缩放")

## 📊 可视化2：交互式散点图

In [ ]:
# 创建交互式散点图：温度-NDVI关系
fig = px.scatter(df, 
                 x='NDVI', 
                 y='温度(°C)', 
                 color='城市',
                 size='温度(°C)',
                 hover_data=['月份'],
                 title='温度-NDVI关系',
                 labels={'NDVI': '归一化植被指数', '温度(°C)': '温度 (°C)'},
                 opacity=0.7)

# 添加趋势线
for city in cities:
    city_data = df[df['城市'] == city]
    z = np.polyfit(city_data['NDVI'], city_data['温度(°C)'], 1)
    p = np.poly1d(z)
    x_range = np.linspace(city_data['NDVI'].min(), city_data['NDVI'].max(), 100)
    
    fig.add_traces(go.Scatter(
        x=x_range,
        y=p(x_range),
        mode='lines',
        name=f'{city} 趋势线',
        line=dict(dash='dash', width=2),
        marker=dict(size=1)
    ))

fig.update_layout(
    hovermode='closest',
    xaxis_title='归一化植被指数',
    yaxis_title='温度 (°C)',
    template='plotly_white'
)

fig.show()

print("✅ 散点图已生成！")
print("\n交互功能:")
print("  - 悬停显示月份信息")
print("  - 气泡大小表示温度")
print("  - 趋势线显示整体关系")

## 🗺️ 可视化3：交互式地图

In [ ]:
# 创建带有坐标的示例数据
city_coords = {
    '北京': {'lon': 116.4, 'lat': 39.9},
    '上海': {'lon': 121.4, 'lat': 31.2},
    '广州': {'lon': 113.3, 'lat': 23.1}
}

# 创建更详细的模拟数据（多个采样点）
map_data = []
for city in cities:
    center = city_coords[city]
    for i in range(20):  # 每个城市20个采样点
        map_data.append({
            '城市': city,
            '经度': center['lon'] + np.random.uniform(-0.2, 0.2),
            '纬度': center['lat'] + np.random.uniform(-0.15, 0.15),
            '温度(°C)': df[df['城市'] == city]['温度(°C)'].mean() + np.random.normal(0, 2),
            'NDVI': df[df['城市'] == city]['NDVI'].mean() + np.random.normal(0, 0.1)
        })

map_df = pd.DataFrame(map_data)

# 创建交互式地图：温度分布
fig = px.scatter_mapbox(map_df,
                          lat='纬度',
                          lon='经度',
                          color='温度(°C)',
                          size='NDVI',
                          hover_name='城市',
                          hover_data=['温度(°C)', 'NDVI'],
                          color_continuous_scale='RdYlBu_r',
                          zoom=4,
                          center={'lat': 35, 'lon': 115},
                          title='城市温度分布图',
                          mapbox_style='open-street-map',
                          labels={'温度(°C)': '温度 (°C)', 'NDVI': '归一化植被指数'})

fig.update_layout(
    margin=dict(l=0, r=0, t=30, b=0),
    template='plotly_white'
)

fig.show()

print("✅ 交互式地图已生成！")
print("\n交互功能:")
print("  - 缩放和平移")
print("  - 悬停显示详细数据")
print("  - 颜色表示温度")
print("  - 气泡大小表示NDVI")

## 📊 可视化4：交互式柱状图

In [ ]:
# 创建交互式柱状图：月度对比
monthly_avg = df.groupby(['月份', '城市']).mean().reset_index()

fig = px.bar(monthly_avg,
             x='月份',
             y='温度(°C)',
             color='城市',
             barmode='group',
             title='月度温度对比',
             labels={'温度(°C)': '温度 (°C)'},
             text='温度(°C)',
             template='plotly_white')

fig.update_traces(texttemplate='%{text:.1f}', textposition='outside')
fig.update_layout(
    xaxis_title='月份',
    yaxis_title='温度 (°C)',
    uniformtext_minsize=8,
    uniformtext_mode='hide'
)

fig.show()

print("✅ 柱状图已生成！")

## 🌡️ 可视化5：热力图

In [ ]:
# 创建热力图数据透视表
heatmap_data = df.pivot(index='城市', columns='月份', values='温度(°C)')

fig = px.imshow(heatmap_data,
                  labels=dict(x="月份", y="城市", color="温度 (°C)"),
                  x=heatmap_data.columns,
                  y=heatmap_data.index,
                  color_continuous_scale='RdYlBu_r',
                  title='城市-月份温度热力图',
                  aspect='auto')

fig.update_layout(
    template='plotly_white',
    xaxis_title='月份',
    yaxis_title='城市'
)

fig.show()

print("✅ 热力图已生成！")
print("\n交互功能:")
print("  - 悬停查看精确数值")
print("  - 颜色深浅表示温度高低")

## 📈 可视化6：箱线图

In [ ]:
# 创建箱线图：数据分布
fig = px.box(df, 
              x='城市', 
              y='温度(°C)', 
              color='城市',
              title='城市温度分布箱线图',
              labels={'温度(°C)': '温度 (°C)'},
              template='plotly_white')

fig.update_traces(boxmean=True)  # 显示均值

fig.update_layout(
    yaxis_title='温度 (°C)',
    showlegend=False
)

fig.show()

print("✅ 箱线图已生成！")
print("\n交互功能:")
print("  - 显示中位数、四分位数")
print("  - 识别异常值")
print("  - 对比不同城市")

## 📊 可视化7：时间序列动画

In [ ]:
# 创建时间序列动画：温度和NDVI的月度变化
fig = px.scatter(df,
                 x='NDVI',
                 y='温度(°C)',
                 animation_frame='月份',
                 color='城市',
                 size='温度(°C)',
                 hover_name='城市',
                 hover_data=['月份', '温度(°C)', 'NDVI'],
                 title='温度-NDVI关系月度变化',
                 labels={'NDVI': '归一化植被指数', '温度(°C)': '温度 (°C)'},
                 range_x=[0, 1],
                 range_y=[0, 35],
                 template='plotly_white')

# 调整动画速度
fig.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 1000
fig.layout.updatemenus[0].buttons[0].args[1]['transition']['duration'] = 500

fig.show()

print("✅ 时间序列动画已生成！")
print("\n交互功能:")
print("  - 点击播放按钮观看动画")
print("  - 拖动滑块查看特定月份")
print("  - 观察数据随时间的演变")

## 💾 导出交互式图表

In [ ]:
# 导出图表为HTML
html_file = 'interactive_plots.html'

# 创建多个图表并保存
from plotly.subplots import make_subplots

# 创建子图
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('温度趋势', 'NDVI趋势', '温度分布', '相关关系'),
    specs=[[{'secondary_y': False}, {'secondary_y': False}],
             [{'secondary_y': False}, {'secondary_y': False}]]
)

# 添加折线图
for city in cities:
    city_data = df[df['城市'] == city]
    fig.add_trace(
        go.Scatter(x=city_data['月份'], y=city_data['温度(°C)'], 
                   name=f'{city} 温度', mode='lines+markers'),
        row=1, col=1
    )

fig.update_layout(
    title_text='城市环境数据综合分析',
    showlegend=True,
    height=600,
    template='plotly_white'
)

# 保存为HTML
fig.write_html(html_file)

print(f"✅ 图表已导出为HTML: {html_file}")
print("\n使用方法:")
print("  - 在浏览器中打开HTML文件")
print("  - 所有交互功能都保留")
print("  - 可以分享给他人")
print("  - 可以嵌入到网页中")

---

## 🎉 总结

### 你学会了创建：

1. ✅ **交互式折线图** - 趋势分析
2. ✅ **交互式散点图** - 关系分析
3. ✅ **交互式地图** - 空间分布
4. ✅ **交互式柱状图** - 对比分析
5. ✅ **热力图** - 模式识别
6. ✅ **箱线图** - 分布分析
7. ✅ **时间序列动画** - 动态变化

### 🚀 优势：

- 📱 支持缩放、平移
- 🎯 悬停显示详情
- 🔍 动态筛选
- 💾 可导出HTML
- 🌐 可嵌入网页

### 📚 更多资源：

- Plotly文档: https://plotly.com/python/
- Plotly Express: https://plotly.com/python/plotly-express/
- 示例画廊: https://plotly.com/python/

---

**继续探索: 🎨**
- 尝试不同的颜色主题
- 创建3D可视化
- 添加更多交互控件
- 自定义图表样式